In [1]:
TOOLS = [
    {
        "type": "function",
        "name": "open_application",
        "description": "Open a desktop application.",
        "parameters": {
            "type": "object",
            "properties": {"app": {"type": "string", "description": "Application name, e.g. chrome, notepad, vscode"}},
            "required": ["app"],
        },
    },
    {
        "type": "function",
        "name": "search",
        "description": "Search a platform (web, YouTube, etc.) for a query.",
        "parameters": {
            "type": "object",
            "properties": {
                "platform": {"type": "string", "description": "e.g. google, youtube, spotify"},
                "query": {"type": "string", "description": "The search text"},
            },
            "required": ["platform", "query"],
        },
    },
    {
        "type": "function",
        "name": "create_folder",
        "description": "Create a new folder.",
        "parameters": {
            "type": "object",
            "properties": {"path": {"type": "string", "description": "Folder name or path"}},
            "required": ["path"],
        },
    },
    {
        "type": "function",
        "name": "move_file",
        "description": "Move a file or folder to a new location.",
        "parameters": {
            "type": "object",
            "properties": {
                "source": {"type": "string"},
                "destination": {"type": "string"},
            },
            "required": ["source", "destination"],
        },
    },
    {
        "type": "function",
        "name": "run_command",
        "description": "Run a shell command. Reserved for future use.",
        "parameters": {
            "type": "object",
            "properties": {"command": {"type": "string"}},
            "required": ["command"],
        },
    },
    {
        "type": "function",
        "name": "open_website",
        "description": "Open a specific URL directly. Only use when the user gives an explicit URL.",
        "parameters": {
            "type": "object",
            "properties": {"url": {"type": "string"}},
            "required": ["url"],
        },
    },
    {
        "type": "function",
        "name": "gui_action",
        "description": "Perform a GUI action such as typing text. Reserved for future use.",
        "parameters": {
            "type": "object",
            "properties": {
                "gui_command": {"type": "string", "description": "e.g. type_text, click, scroll"},
                "text": {"type": "string"},
            },
            "required": ["gui_command"],
        },
    },
    {
        "type": "function",
        "name": "request_clarification",
        "description": "Call this instead of any action tool when the user's request is ambiguous or missing required information. Do not guess.",
        "parameters": {
            "type": "object",
            "properties": {"question": {"type": "string", "description": "A specific question to ask the user"}},
            "required": ["question"],
        },
    },
    {
        "type": "function",
        "name": "task_complete",
        "description": "Call this once all necessary steps have been proposed and no more actions are needed.",
        "parameters": {
            "type": "object",
            "properties": {"summary": {"type": "string", "description": "Short summary of what was planned"}},
            "required": ["summary"],
        },
    },
]

In [2]:
def mock_execute_tool(name: str, arguments: dict) -> dict:
    """Simulates a tool result, standing in for Person 2/3's real Action Router."""
    return {"status": "success", "tool": name, "arguments": arguments, "note": "MOCKED result — no real execution happened."}


def mock_clarification_answer(question: str) -> str:
    """Simulates a user's answer to a clarification question, for automated testing."""
    return f"[MOCKED USER ANSWER to: '{question}'] — placeholder answer for testing."

In [4]:
import os

from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("Gemini_api")

In [5]:
from google import genai
client = genai.Client(api_key=api_key)

In [7]:
SYSTEM_INSTRUCTIONS = (
    "You are the task-planning brain for a laptop automation tool. "
    "Given a user's request, call the appropriate tools one at a time to build a plan. "
    "If required information is missing or the request is ambiguous, call request_clarification "
    "instead of guessing. Once the plan is complete, call task_complete."
)

In [ ]:

GENERATION_CONFIG = {
    "thinking_level": "low",
    "max_output_tokens": 4096,
}

In [ ]:
MAX_TURNS = 10

def run_agent_loop(user_query: str, max_turns: int = MAX_TURNS):
    transcript = []
    interaction = client.interactions.create(
        model="gemini-3.5-flash",
        system_instruction=SYSTEM_INSTRUCTIONS,
        input=user_query,
        tools=TOOLS,
        generation_config=GENERATION_CONFIG,
    )

    for turn in range(max_turns):
        function_calls = [o for o in interaction.outputs if o.type == "function_call"]

        if not function_calls:
            # Model responded with text instead of a tool call — treat as unexpected, log and stop.
            transcript.append({"turn": turn, "event": "unexpected_text_response", "interaction": interaction})
            break

        function_results_input = []
        stop_loop = False

        for call in function_calls:
            if call.name == "task_complete":
                transcript.append({"turn": turn, "event": "task_complete", "arguments": call.arguments})
                stop_loop = True
                break

            elif call.name == "request_clarification":
                question = call.arguments.get("question")
                answer = mock_clarification_answer(question)
                transcript.append({"turn": turn, "event": "clarification", "question": question, "mock_answer": answer})
                function_results_input.append({
                    "type": "function_result",
                    "call_id": call.id,
                    "name": call.name,
                    "result": {"user_answer": answer},
                })

            else:
                result = mock_execute_tool(call.name, call.arguments)
                transcript.append({"turn": turn, "event": "action", "tool": call.name, "arguments": call.arguments, "mock_result": result})
                function_results_input.append({
                    "type": "function_result",
                    "call_id": call.id,
                    "name": call.name,
                    "result": result,
                })

        if stop_loop:
            break

        interaction = client.interactions.create(
            model="gemini-3.5-flash",
            input=function_results_input,
            tools=TOOLS,
            generation_config=GENERATION_CONFIG,
            previous_interaction_id=interaction.id,
        )
    else:
        transcript.append({"turn": max_turns, "event": "max_turns_exceeded"})

    return transcript

In [9]:

test_inputs_v2 = [
    "Open Chrome and search Google for Python documentation",  # control: clear, multi-step
    "Open my project",                                          # ambiguity -> request_clarification
    "Open Chrome and search for Python documentation, also open my RAG project",  # mixed: one clear step, one ambiguous
    "Clean up my downloads folder",                             # the "never guess on destructive action" case
    "Search YouTube",                                           # missing required field (query)
]

In [13]:
interaction = client.interactions.create(
    model="gemini-3.5-flash",
    system_instruction=SYSTEM_INSTRUCTIONS,
    input="Open Chrome and search Google for Python documentation",
    tools=TOOLS,
    generation_config=GENERATION_CONFIG,
)

print(type(interaction))
print([attr for attr in dir(interaction) if not attr.startswith("_")])

<class 'google.genai._gaos.types.interactions.interaction.Interaction'>
['agent', 'agent_config', 'construct', 'copy', 'created', 'dict', 'environment', 'environment_id', 'errors', 'from_orm', 'generation_config', 'id', 'input', 'json', 'labels', 'model', 'model_computed_fields', 'model_config', 'model_construct', 'model_copy', 'model_dump', 'model_dump_json', 'model_extra', 'model_fields', 'model_fields_set', 'model_json_schema', 'model_parametrized_name', 'model_post_init', 'model_rebuild', 'model_validate', 'model_validate_json', 'model_validate_strings', 'output_audio', 'output_image', 'output_text', 'output_video', 'parse_file', 'parse_obj', 'parse_raw', 'previous_interaction_id', 'response_format', 'response_mime_type', 'response_modalities', 'safety_settings', 'schema', 'schema_json', 'service_tier', 'status', 'steps', 'system_instruction', 'to_dict', 'to_json', 'tools', 'update_forward_refs', 'updated', 'usage', 'validate', 'webhook_config']


In [14]:
for step in interaction.steps:
    print(step.type if hasattr(step, "type") else type(step))
    print(step)
    print("---")

thought
signature='Et4JCtsJARFNMg8Hkz/vjE1IT1L9EqCW0zBPwNktNrPgRpaH9MCdbJfhQ0arFGJMXkLMGUVc6jb7vxS4I4VvSOCwiEOCbrvul7erp5OGU+HunWiLYiziaJIUldbFy0HgFveaB6Oj/2PwU8hnI9TKLkuzfKn5Kpv8qRKfxQeHJylgy1M9BuEyHBygg8oyLYdMgPs6ZnYNhY4J2bBmdeGdlsFXZ+O+xnTykUyBJaJAn4sFh++xjlu/l5fuOZWrxrDfkvoWwD+87MGaD0LNKJcikngap49L/Rqt6SOsMU2SFbTxI6ObvlA7MigaEt7T95S6Jvx5AXNH+VvWvRQ2OYN9y6aDSRoLu4Uq4i5r19cFIo3PaE0SJP+6xrpEB2NqNEN9xA95vdBo7UVlwtk39ltUCATQxtwsRG6SXtlRhbOm9RLLSg2WCajy6ntnscdKmC3TOBve2YwsCwtdXiWVO/VJpmkh+480qowzUrPaGT1I+MhITPo1Eyiw2qo7EJ1kN7PoiKXeWT9R2I9GggBDAtleKzHC1FwyBSJYwOzUqQkK9IBKyp1FPxqxSdToCiQjKISsFrDframeQ5w0vB2boVi396n1hl/i+yEpckJgPR+ENARaD0LgAI221rO+VpVgRtEhnlE4K+K8oPtJHvYFvTVVjibZ6aY9h1ZgkUY5Yv9J7lJKubRzKAAk6njWNSaC52UpW5aFlVrLfwf0tTpwLkQDKTJqG7ZrFw0fBO5AO/Cax540y47dI49OWDxclF871TrOLFqO1SBxWkuNMYYeG4E1YfidZiMr8hwbytdKi8b/FZaCbaEiwnuBSzLJ/AWrBjEVye6FoLCm26viA7Iebao738FzqYcmDIP3PsoCg2wcmnvqpOvnQtpstnSnQzwpNY03WZM8j+Ms7rxjSGz0YFu6jY/Qz/jKwW3K+q0z8erntmGH8f2o74GsQEjyUyv9Mwc3QBO+p7/r/vTZ5DzFKWeJi

In [17]:
import time

def run_agent_loop(user_query: str, max_turns: int = MAX_TURNS, turn_delay: int = 13):
    transcript = []
    interaction = client.interactions.create(
        model="gemini-3.5-flash",
        system_instruction=SYSTEM_INSTRUCTIONS,
        input=user_query,
        tools=TOOLS,
        generation_config=GENERATION_CONFIG,
    )

    for turn in range(max_turns):
        function_calls = [s for s in interaction.steps if s.type == "function_call"]

        if not function_calls:
            transcript.append({"turn": turn, "event": "unexpected_no_function_call", "steps": interaction.steps})
            break

        function_results_input = []
        stop_loop = False

        for call in function_calls:
            if call.name == "task_complete":
                transcript.append({"turn": turn, "event": "task_complete", "arguments": call.arguments})
                stop_loop = True
                break
            elif call.name == "request_clarification":
                question = call.arguments.get("question")
                answer = mock_clarification_answer(question)
                transcript.append({"turn": turn, "event": "clarification", "question": question, "mock_answer": answer})
                function_results_input.append({
                    "type": "function_result", "call_id": call.id, "name": call.name,
                    "result": {"user_answer": answer},
                })
            else:
                result = mock_execute_tool(call.name, call.arguments)
                transcript.append({"turn": turn, "event": "action", "tool": call.name, "arguments": call.arguments, "mock_result": result})
                function_results_input.append({
                    "type": "function_result", "call_id": call.id, "name": call.name, "result": result,
                })

        if stop_loop:
            break

        time.sleep(turn_delay)
        interaction = client.interactions.create(
            model="gemini-3.5-flash",
            input=function_results_input,
            tools=TOOLS,
            generation_config=GENERATION_CONFIG,
            previous_interaction_id=interaction.id,
        )
    else:
        transcript.append({"turn": max_turns, "event": "max_turns_exceeded"})

    return transcript

In [18]:
all_results = {}
for query in test_inputs_v2:
    print(f"\n=== Running: {query} ===")
    try:
        all_results[query] = run_agent_loop(query)
    except Exception as e:
        all_results[query] = [{"event": "error", "error": str(e)}]
    time.sleep(15)  # gap between full test cases, on top of per-turn delay



=== Running: Open Chrome and search Google for Python documentation ===

=== Running: Open my project ===

=== Running: Open Chrome and search for Python documentation, also open my RAG project ===

=== Running: Clean up my downloads folder ===

=== Running: Search YouTube ===


In [19]:

for query, transcript in all_results.items():
    print(f"\n--- Input: {query} ---")
    for step in transcript:
        print(f"  {step}")


--- Input: Open Chrome and search Google for Python documentation ---
  {'turn': 0, 'event': 'action', 'tool': 'open_application', 'arguments': {'app': 'chrome'}, 'mock_result': {'status': 'success', 'tool': 'open_application', 'arguments': {'app': 'chrome'}, 'note': 'MOCKED result — no real execution happened.'}}
  {'turn': 1, 'event': 'action', 'tool': 'search', 'arguments': {'query': 'Python documentation', 'platform': 'google'}, 'mock_result': {'status': 'success', 'tool': 'search', 'arguments': {'query': 'Python documentation', 'platform': 'google'}, 'note': 'MOCKED result — no real execution happened.'}}
  {'turn': 2, 'event': 'task_complete', 'arguments': {'summary': 'Opened Chrome and searched Google for Python documentation.'}}

--- Input: Open my project ---
  {'turn': 0, 'event': 'clarification', 'question': 'Could you please specify which project you would like to open, its location, or the application you want to open it with?', 'mock_answer': "[MOCKED USER ANSWER to: 'Co